# Task 2 — Exploratory Data Analysis

This notebook explores the reshaped ATP match data (`atp_matches_reshaped.csv`) before feature engineering and modelling. Each row is one match with a randomly assigned `player` vs. `opp` perspective (no winner/loser leakage).

**Definitions used throughout:**
- **Favorite** = the player with the better ATP ranking (lower rank number)
- **Favorite wins** = the higher-ranked player won the match
- **Baseline** = the simple rule “always pick the higher-ranked player” — our benchmark for supervised learning (Task 5)

In [23]:
import pandas as pd

from helpers.featureAnalysis import (
    FORM_WINDOW,
    LONG_MATCH_MINUTES,
    add_player_history,
    ordered_rounds,
    print_section,
    print_table,
    ranked_matches,
)
from helpers.loading import load_reshaped_data

### Dataset overview

**What we examine:** Size of the dataset and completeness of ATP ranking information.

**Result:** 112,579 matches (1991–2026); 97.0% of rows have both `player_rank` and `opp_rank`.

**Interpretation:** Nearly all matches are usable for the ranking baseline. The ~3% without ranks (e.g. unranked qualifiers) will need to be excluded or handled separately in later modelling steps.

In [19]:
df = load_reshaped_data()
df["year"] = df["tourney_date"] // 10000
ranked = ranked_matches(df)
history = add_player_history(df)

print_section("Dataset overview")
print(f"Rows: {len(df):,}  |  Years: {int(df['year'].min())}-{int(df['year'].max())}")
print(f"Matches with both ranks: {len(ranked):,} ({len(ranked) / len(df):.1%} of rows)")


Loading reshaped data from /Users/franz/Coding/Uni/DAI-Tennis-Prediction/data/atp_matches_reshaped.csv

Dataset overview
Rows: 112,579  |  Years: 1991-2026
Matches with both ranks: 109,252 (97.0% of rows)


### 1. Favorite win rate by surface

**What we examine:** Does the higher-ranked player win more often on certain court surfaces?

**Result:** Favorite win rates are similar across surfaces — Hard 65.8%, Grass 64.8%, Clay 64.4%, Carpet 64.0%.

**Interpretation:** Differences are small (roughly 64–66%). The ranking is about equally informative on every surface; Hard is marginally more predictable, Clay is not dramatically more upset-prone. Surface is still worth including as a feature later, but it is unlikely to change the overall predictability story on its own.

In [21]:
print_section("1. Favorite win rate by surface")
by_surface = (
    ranked.groupby("surface", dropna=False)["favorite_wins"]
    .agg(matches="count", favorite_win_rate="mean")
    .sort_values("favorite_win_rate", ascending=False)
)
by_surface["favorite_win_rate"] = (by_surface["favorite_win_rate"] * 100).round(1)
print_table(by_surface, "Higher-ranked player win rate (%) by surface")



1. Favorite win rate by surface

Higher-ranked player win rate (%) by surface
         matches  favorite_win_rate
surface                            
NaN           36               72.2
Hard       55364               65.8
Grass      10681               64.8
Clay       36285               64.4
Carpet      6886               64.0


### 2. Favorite win rate by round

**What we examine:** Does predictability change by tournament round?

**Result:** Round Robin ~70.3%; main-draw rounds (R128–QF) cluster around 64–66%; semi-finals are lowest at 62.4%.

**Interpretation:**
- **Round Robin (70.3%):** Higher favourite win rate — likely reflects clearer quality gaps in group-stage match-ups or a different player mix.
- **Early/mid rounds (R128–QF):** Stable at ~64–66%, close to the overall baseline.
- **Semi-finals (62.4%):** Slightly lower — late-stage matches tend to pit similarly strong players against each other, so upsets become more likely. Estimates for F, ER, and BR are based on small samples and should be read with caution.

In [28]:
print_section("2. Favorite win rate by round")
by_round = (
    ranked.groupby("round", dropna=False)["favorite_wins"]
    .agg(matches="count", favorite_win_rate="mean")
)
by_round = by_round.reindex(ordered_rounds(by_round.index))
by_round["favorite_win_rate"] = (by_round["favorite_win_rate"] * 100).round(1)
print_table(by_round, "Higher-ranked player win rate (%) by round")


2. Favorite win rate by round

Higher-ranked player win rate (%) by round
       matches  favorite_win_rate
round                            
RR        9099               70.3
R128     11121               65.3
R64      16869               64.4
R32      35896               64.3
R16      19160               65.9
QF        9636               65.0
SF        4923               62.4
F         2506               63.1
ER          32               59.4
BR          10               40.0


### 3. Ranking-gap baseline

**What we examine:** The core benchmark from our research question — *does the higher-ranked player win?* This is the target any supervised model (Task 5) must beat.

**Result:** Overall baseline accuracy = **65.1%** (71,154 / 109,252 ranked matches). This matches the proposal expectation of ~65–70%.

**By rank gap:**

| Gap | Favourite win rate |
|-----|-------------------|
| 1–5 | 52.8% |
| 6–10 | 55.3% |
| 11–20 | 58.6% |
| 21–50 | 63.0% |
| 51–100 | 69.0% |
| 100+ | 74.4% |

**Interpretation:** There is a clear monotonic pattern — the larger the ranking gap, the more often the favourite wins. In **close match-ups (gap 1–5)** the baseline is almost a coin flip (~53%). At **gap 100+** the favourite wins roughly three out of four times. The ranking is therefore not a uniform signal: it is strong when quality differences are large and weak when top players meet. That is exactly where form, fatigue, and playing-style features may add the most value.

In [25]:
print_section("3. Ranking-gap baseline")
overall_baseline = ranked["favorite_wins"].mean()
print(
    f"Higher-ranked player wins: {overall_baseline:.1%} "
    f"({ranked['favorite_wins'].sum():,} / {len(ranked):,} ranked matches)"
)

gap_bins = [0, 5, 10, 20, 50, 100, 10_000]
gap_labels = ["1-5", "6-10", "11-20", "21-50", "51-100", "100+"]
ranked["gap_bucket"] = pd.cut(
    ranked["rank_gap"],
    bins=gap_bins,
    labels=gap_labels,
    right=True,
)
by_gap = (
    ranked.groupby("gap_bucket", observed=True)["favorite_wins"]
    .agg(matches="count", favorite_win_rate="mean")
)
by_gap["favorite_win_rate"] = (by_gap["favorite_win_rate"] * 100).round(1)
print_table(by_gap, "Baseline accuracy by absolute rank gap")



3. Ranking-gap baseline
Higher-ranked player wins: 65.1% (71,154 / 109,252 ranked matches)

Baseline accuracy by absolute rank gap
            matches  favorite_win_rate
gap_bucket                            
1-5            7496               52.8
6-10           7213               55.3
11-20         13407               58.6
21-50         31915               63.0
51-100        25363               69.0
100+          23858               74.4


### 4. Fatigue prevalence

**What we examine:** How often does **fatigue** occur (previous match ≥ 180 minutes) and is there a descriptive link to win probability? This is *not* causal yet — Divine's DoWhy analysis (Task 4) will test that properly.

**Result:**
- 86.1% of rows have a prior match in the dataset
- Long previous match: **4.2%** of all rows (4.8% among rows with a prior match)
- Win rate after normal previous match: **50.8%** vs. **52.4%** after a long one
- Median rest since previous match: **10 days**; **31.2%** of matches follow ≤1 rest day

**Interpretation:**
- Fatigue events are **rare** (~1 in 24 matches with history). The feature is sparse but not negligible; there should still be enough cases for causal analysis.
- The slightly *higher* win rate after a long match is **not** evidence that fatigue helps. It almost certainly reflects **confounding** — e.g. fitter or higher-quality players play longer matches and win more often. Ranking, surface, and round are not controlled here.
- **Short rest (≤1 day, 31%)** is relatively common. A continuous `rest_days` feature may be as informative as the binary `long_prev_match` flag.

In [26]:
print_section("4. Fatigue prevalence")
has_prev_match = history["prev_minutes"].notna()
long_prev = history["long_prev_match"].fillna(False)

print(f"Rows with a prior match in dataset: {has_prev_match.mean():.1%}")
print(
    f"Rows with long previous match (>={LONG_MATCH_MINUTES} min): "
    f"{long_prev.mean():.1%} of all rows, "
    f"{long_prev[has_prev_match].mean():.1%} of rows with prior match"
)

fatigue_outcomes = history.loc[has_prev_match].groupby("long_prev_match")["win"].agg(
    matches="count", win_rate="mean"
)
fatigue_outcomes["win_rate"] = (fatigue_outcomes["win_rate"] * 100).round(1)
print_table(
    fatigue_outcomes,
    "Player win rate after normal vs. long previous match (descriptive only)",
)

rest = history.loc[history["rest_days"].notna(), "rest_days"]
rest_summary = rest.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(1)
print("\nRest days since previous match:")
print(rest_summary.to_string())

short_rest = (rest <= 1).mean()
print(f"\nShare with <=1 rest day since previous match: {short_rest:.1%}")



4. Fatigue prevalence
Rows with a prior match in dataset: 86.1%
Rows with long previous match (>=180 min): 4.2% of all rows, 4.8% of rows with prior match

Player win rate after normal vs. long previous match (descriptive only)
                 matches  win_rate
long_prev_match                   
False              92248      50.8
True                4700      52.4

Rest days since previous match:
count    109429.0
mean         45.6
std         153.2
min           0.0
25%           0.0
50%          10.0
75%          28.0
90%          84.0
max        5083.0

Share with <=1 rest day since previous match: 31.2%


### 5. Form prevalence

**What we examine:** How widespread is **recent form** (win rate over the prior 5 matches) and does it correlate with match outcomes?

**Result:**
- 90.5% of rows have ≥5 prior matches for rolling form
- Hot form (≥80%): 22.9% of rows; cold form (≤20%): 19.6%
- Win rate by form bucket rises from **42.4%** (0–20%) to **61.6%** (81–100%)

**Interpretation:**
- Form is **widely available** and shows a **strong gradient** — much clearer than the raw fatigue comparison above.
- The ~20 percentage-point spread from cold to hot form suggests `recent_form` is a promising feature for Task 3, especially in **tight ranking match-ups** where the baseline is only ~53%.
- As with fatigue, this is descriptive, not causal: good form correlates with high ranking. The supervised model must separate form from ranking information.

---

### Overall takeaway

| Question | EDA answer |
|----------|-----------|
| Is prediction worthwhile? | Yes — 65.1% baseline leaves ~35% unexplained |
| Where is the baseline weakest? | Close rankings (gap 1–5 → ~53%) |
| Is fatigue worth encoding? | Rare (4.2%); causal effect still open (Task 4) |
| Is form worth encoding? | Yes — strong gradient, broad coverage |

**Next step (Task 3):** Build chronological features — recent form, surface win rate, head-to-head, rest days, and `long_prev_match` — with strict ordering to avoid future leakage.

In [27]:
print_section("5. Form prevalence")
enough_history = history["prior_matches"] >= FORM_WINDOW
print(
    f"Rows with at least {FORM_WINDOW} prior matches for rolling form: "
    f"{enough_history.mean():.1%}"
)

form_summary = history.loc[history["recent_form"].notna(), "recent_form"].describe(
    percentiles=[0.25, 0.5, 0.75]
)
print(f"\nRecent form (win rate over prior {FORM_WINDOW} matches):")
print((form_summary * 100).round(1).to_string())

hot_form = history["recent_form"].ge(0.8)
cold_form = history["recent_form"].le(0.2)
print(f"\nShare with recent form >= 80%: {hot_form.mean():.1%}")
print(f"Share with recent form <= 20%: {cold_form.mean():.1%}")

form_outcomes = history.loc[history["recent_form"].notna()].copy()
form_outcomes["form_bucket"] = pd.cut(
    form_outcomes["recent_form"],
    bins=[-0.01, 0.2, 0.4, 0.6, 0.8, 1.01],
    labels=["0-20%", "21-40%", "41-60%", "61-80%", "81-100%"],
)
by_form = (
    form_outcomes.groupby("form_bucket", observed=True)["win"]
    .agg(matches="count", win_rate="mean")
)
by_form["win_rate"] = (by_form["win_rate"] * 100).round(1)
print_table(by_form, "Player win rate by recent-form bucket (descriptive only)")



5. Form prevalence
Rows with at least 5 prior matches for rolling form: 90.5%

Recent form (win rate over prior 5 matches):
count    10942900.0
mean           51.2
std            25.7
min             0.0
25%            40.0
50%            60.0
75%            60.0
max           100.0

Share with recent form >= 80%: 22.9%
Share with recent form <= 20%: 19.6%

Player win rate by recent-form bucket (descriptive only)
             matches  win_rate
form_bucket                   
0-20%          22028      42.4
21-40%         29310      48.3
41-60%         31612      52.2
61-80%         19134      57.6
81-100%         7345      61.6
